# Export room geometries to .obj

Writes one `.obj` per *distinct room geometry* used by the suite (diffuse/specular variants of the same shoebox size share identical geometry -- only material differs -- so only one is exported per size) to `pra_comparison_assets/room_geometries/`.

Two things matter for the output to be importable as a volumetric mesh (e.g. via CHORAS's gmsh/tetgen pipeline), not just for viewing:

1. **Shared vertices at wall seams.** Each `rectangle` primitive is analytic and independent in mitsuba; naively exporting every wall with its own private 4 vertices leaves numerically-coincident-but-topologically-separate points at every edge two walls share. TetGen's PLC volume mesher chokes on that (`PLC Error: A segment and a facet intersect at a point`) -- it needs a real watertight shell with *shared* vertex indices at shared edges. Fixed by welding vertices into one pool per scenario, rounded to 4 decimals (0.1 mm) -- loose enough to absorb the float32 rotation noise mitsuba's `rotate()` introduces at a shared edge (~1e-6 to ~1e-5 depending on room scale, see `export_scenario_obj`'s docstring), tight enough to never merge two genuinely distinct points at room scale.
2. **Consistent, outward-facing winding.** Each wall's `flip_normals` flag (set per-wall in `test_pyroomacoustics_comparison.py` so every wall of a room faces inward for misuka's own rendering) has to be honored when triangulating, otherwise the exported shell mixes inward- and outward-wound facets -- not a consistent closed manifold, another way to trigger the same PLC error class.

A cheap self-check (`_assert_closed_manifold`) verifies every edge of a primitive-based room is shared by exactly 2 triangles before writing -- exactly the property TetGen needs, checked *before* handing it to gmsh instead of discovering it there.

In [ ]:
import os
from collections import Counter

import mitsuba as mi
mi.set_variant("llvm_ad_acoustic")

import test_pyroomacoustics_comparison as cmp

OUT_DIR = os.path.join("pra_comparison_assets", "room_geometries")
os.makedirs(OUT_DIR, exist_ok=True)

# One entry per distinct geometry -- diffuse/specular scenarios at the same
# size only differ in material (absorption/scattering), not shape, so only
# the diffuse variant is exported for each shoebox size.
GEOMETRY_SCENARIOS = [
    (cmp.scenario_shoebox_diffuse, "shoebox_200m3"),
    (cmp.scenario_shoebox_diffuse_1000, "shoebox_1000m3"),
    (cmp.scenario_shoebox_diffuse_5000, "shoebox_5000m3"),
    (cmp.scenario_l_room, "l_room_1000m3"),
    (cmp.scenario_coincident_reflections, "coincident_reflections"),
    (cmp.scenario_flutter_corridor, "flutter_corridor"),
    (cmp.scenario_auditorium_complex, "auditorium_complex"),
]

In [ ]:
def _to_xyz(p):
    """mitsuba Point3f -> (x, y, z) floats, for both scalar and JIT-vectorized point types."""
    try:
        return float(p.x), float(p.y), float(p.z)
    except TypeError:
        return float(p.x[0]), float(p.y[0]), float(p.z[0])


def _to_ijk(f):
    """mitsuba Array3u (face indices) -> (i, j, k) ints, scalar or JIT-vectorized."""
    try:
        return int(f.x), int(f.y), int(f.z)
    except TypeError:
        return int(f.x[0]), int(f.y[0]), int(f.z[0])


def _to_choras_axes(p):
    """Remap misuka's Z-up frame (floor at z=0, footprint in X/Y >= 0) to the
    convention CHORAS's own example rooms actually use: Y-up, footprint in
    X (>= 0) and Z (<= 0) -- verified against example_geometries/
    Room2215_simple.obj, Room2215_withAbs.obj and MeasurementRoom.obj (all
    three: x in [0, ...], y in [0, room_height], z in [-depth, 0]).
    (x, y, z) -> (x, z, -y) is a proper rotation (determinant +1), so it
    doesn't touch winding/normal direction -- flip_normals handling in
    _rectangle_dict_quad still applies before this remap, unchanged.
    """
    x, y, z = p
    return (x, z, -y)


def _rectangle_dict_quad(shape_dict):
    """World-space corners of a misuka 'rectangle' primitive (local unit
    square at z=0), in consistent winding order honoring 'flip_normals' --
    reversing the quad's vertex order reverses its winding/normal direction,
    exactly what flip_normals does for misuka's own renderer. Returned in
    CHORAS's axis convention (see _to_choras_axes).
    """
    to_world = shape_dict["to_world"]
    local_corners = [(-1, -1, 0), (1, -1, 0), (1, 1, 0), (-1, 1, 0)]
    corners = [_to_xyz(to_world @ mi.ScalarPoint3f(*c)) for c in local_corners]
    if shape_dict.get("flip_normals", False):
        corners = corners[::-1]
    return [_to_choras_axes(c) for c in corners]


def _assert_closed_manifold(groups, scenario_name):
    """Every undirected edge of a closed room shell must be shared by exactly
    2 triangles. Catches unwelded/duplicate vertices and winding mistakes --
    exactly the property a gmsh/tetgen PLC volume mesher needs -- before they
    surface as an opaque meshing error instead of here.
    """
    edge_count = Counter()
    for _, faces in groups:
        for a, b, c in faces:
            for u, v in ((a, b), (b, c), (c, a)):
                edge_count[frozenset((u, v))] += 1
    bad = {e: n for e, n in edge_count.items() if n != 2}
    if bad:
        print(f"WARNING [{scenario_name}]: {len(bad)} edge(s) not shared by exactly 2 "
              f"triangles (non-manifold/open shell) -- e.g. {list(bad.items())[:3]}")
    return not bad


def export_scenario_obj(scenario, out_path, ndigits=4):
    # ndigits=4 (0.1 mm) rather than a tighter round-trip tolerance: mitsuba's
    # rotate() computes cos/sin in float32, so two walls that should meet
    # exactly at a shared edge can land ~1e-6 to ~1e-5 apart in world space
    # depending on room scale (verified: shoebox_1000m3's rotated walls
    # land ~9.5e-7 off the axis-aligned walls' shared edge coordinate) --
    # rounding to 6 decimals was tighter than that noise floor and silently
    # left seams unwelded (non-manifold shell, the actual PLC-error cause).
    # 4 decimals safely absorbs it while staying far below any real
    # geometric feature at room scale.
    misuka_scene = scenario["misuka_scene"]

    vertex_index = {}  # rounded (x, y, z) -> 1-based OBJ vertex index (welds shared seams)
    vertices = []
    groups = []  # (name, [(i, j, k), ...]) with i/j/k 1-based indices into `vertices`

    def add_vertex(p):
        key = tuple(round(c, ndigits) for c in p)
        idx = vertex_index.get(key)
        if idx is None:
            idx = len(vertices) + 1
            vertex_index[key] = idx
            vertices.append(p)
        return idx

    src = mic = None
    if isinstance(misuka_scene, dict):
        src = _to_choras_axes(misuka_scene["emitter"]["center"])
        mic = _to_choras_axes(misuka_scene["mic"]["origin"])
        for name, shape in misuka_scene.items():
            if not isinstance(shape, dict) or shape.get("type") != "rectangle":
                continue  # skips "type", "emitter" (sphere), "mic" (microphone)
            idx = [add_vertex(c) for c in _rectangle_dict_quad(shape)]
            faces = [(idx[0], idx[1], idx[2]), (idx[0], idx[2], idx[3])]
            groups.append((name, faces))
    else:
        # already-loaded mi.Scene (auditorium_complex): real Mesh shapes from PLY
        for i, shape in enumerate(misuka_scene.shapes()):
            if not isinstance(shape, mi.Mesh):
                continue
            local_idx = [add_vertex(_to_choras_axes(_to_xyz(shape.vertex_position(v))))
                         for v in range(shape.vertex_count())]
            faces = [tuple(local_idx[c] for c in _to_ijk(shape.face_indices(f)))
                     for f in range(shape.face_count())]
            groups.append((shape.id() or f"part_{i}", faces))

    _assert_closed_manifold(groups, scenario["name"])

    lines = [f"# {scenario['name']}"]
    if src is not None:
        lines.append(f"# source: {src[0]} {src[1]} {src[2]}")
        lines.append(f"# mic:    {mic[0]} {mic[1]} {mic[2]}")
    lines += [f"v {x:.6f} {y:.6f} {z:.6f}" for x, y, z in vertices]
    for name, faces in groups:
        lines.append(f"o {name}")
        lines += [f"f {a} {b} {c}" for a, b, c in faces]

    with open(out_path, "w") as f:
        f.write("\n".join(lines) + "\n")
    return len(vertices)

In [ ]:
for scenario_fn, geometry_name in GEOMETRY_SCENARIOS:
    scenario = scenario_fn()
    out_path = os.path.join(OUT_DIR, f"{geometry_name}.obj")
    n_verts = export_scenario_obj(scenario, out_path)
    print(f"wrote {out_path} ({n_verts} unique vertices)")